# 🥇 Gold Layer — Business Insights from Loan Transactions

**What we're doing:** Creating business-ready summaries from clean Silver data.

Gold tables power dashboards, executive reports, and KPIs.

In [ ]:
# Import PySpark SQL functions needed for Gold-layer aggregations and rounded metrics
from pyspark.sql import functions as F

# Load the curated Silver Delta table as the source for business-ready Gold reporting
df_silver = spark.table('silver_loan_transactions')
# Count the Silver input rows so the Gold summary starts from the expected cleaned dataset
print(f'Silver records available: {df_silver.count()}')

In [ ]:
# Group the Silver data by branch, month, and transaction type to match the Gold reporting grain
# Calculate transaction counts, total volume, and average amount for each business summary group
df_gold = df_silver.groupBy('Branch', 'Month', 'TransactionType') \
    .agg(
        F.count('TransactionID').alias('TransactionCount'),
        F.round(F.sum('Amount'), 2).alias('TotalAmount'),
        F.round(F.avg('Amount'), 2).alias('AvgAmount')
    ).orderBy('Month', 'Branch')

# Preview the Gold aggregation output before saving it for dashboard and SQL reporting use
df_gold.show(20)


In [ ]:
# Write the aggregated Gold summary to a Delta table for Power BI and executive reporting
df_gold.write.format('delta').mode('overwrite').saveAsTable('gold_loan_summary')
# Confirm that the Gold reporting table has been refreshed successfully
print('✅ Gold table saved — ready for Power BI and dashboards!')


In [ ]:
%%sql
-- Roll up Gold metrics by branch to identify which team handled the most loan volume
SELECT Branch,
       SUM(TransactionCount) AS TotalTransactions,
       ROUND(SUM(TotalAmount), 2) AS TotalVolume
FROM gold_loan_summary
GROUP BY Branch
ORDER BY TotalVolume DESC


In [ ]:
%%sql
-- Compare monthly Gold totals for each transaction type to track payment and disbursement trends
SELECT Month, TransactionType,
       SUM(TransactionCount) AS Transactions,
       ROUND(SUM(TotalAmount), 2) AS Volume
FROM gold_loan_summary
GROUP BY Month, TransactionType
ORDER BY Month
